In [1]:
# TODO: For amounts calculate the confidence intervals for transactions above the average for a particular date. DONE
# TODO: For the location calculate the fraud rate per 100k. DONE
# TODO: Risk rate for location. DONE
# TODO: Factor in for seasonality

In [2]:
# Import packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import seaborn.objects as so
import datetime as dt
from datetime import datetime
import os
# Change directory to the data location
os.chdir('../../ml_data')

In [3]:
# Read the data
df = pd.read_csv('credit_card_fraud_dataset.csv')
fraud_rates_cities = pd.read_csv('fraud_rates_city.csv')

In [4]:
# Drop unnecessary columns
fraud_rates_cities=fraud_rates_cities.drop(columns=['NotFraud', 'Fraud', 'Population'])

In [5]:
# Parameters
bandwidth_plus = 1.25
bandwidth_minus = 0.75

In [6]:
# Create Fraudulent MerchantID DF
fraud = df[df['IsFraud']==1].groupby(by='MerchantID').count().reset_index()
fraud = fraud[['MerchantID', 'IsFraud']]
fraud = fraud.rename(columns={'IsFraud':'FraudCount'})
fraud

,MerchantID,FraudCount
0,1,1
1,3,1
2,4,2
3,5,1
4,6,3
...,...,...
623,992,1
624,997,1
625,998,1
626,999,1


In [7]:
# Split Datetime
df["Year"] =(df["TransactionDate"].astype(str).str.split(" ").str[0]).str.split('-').str[0]
df["Month"]=(df["TransactionDate"].astype(str).str.split(" ").str[0]).str.split('-').str[1]
df["Day"]=(df["TransactionDate"].astype(str).str.split(" ").str[0]).str.split('-').str[2]
df['hour']=(df["TransactionDate"].astype(str).str.split(" ").str[1]).str.split(':').str[0]
df["Year-Month"] = df["Year"]+'-'+df["Month"]

In [8]:
# Create Threshold DF
monay_mean = df[["Amount", "Year-Month"]]
monay_mean= monay_mean.rename(columns={'Amount':'Mean'})
monay_mean = monay_mean.groupby(by=['Year-Month']).mean().reset_index()
monay_mean

,Year-Month,Mean
0,2023-10,2473.868349
1,2023-11,2460.981078
2,2023-12,2511.422951
3,2024-01,2496.120104
4,2024-02,2493.052607
5,2024-03,2516.489406
6,2024-04,2480.872180
7,2024-05,2500.726202
8,2024-06,2468.091999
9,2024-07,2508.055940


In [9]:
df = pd.merge(df, monay_mean, on='Year-Month')
df.head()

,TransactionID,TransactionDate,Amount,MerchantID,TransactionType,Location,IsFraud,Year,Month,Day,hour,Year-Month,Mean
0,1,2024-04-03 14:15:35.462794,4189.27,688,refund,San Antonio,0,2024,04,03,14,2024-04,2480.872180
1,2,2024-03-19 13:20:35.462824,2659.71,109,refund,Dallas,0,2024,03,19,13,2024-03,2516.489406
2,3,2024-01-08 10:08:35.462834,784.00,394,purchase,New York,0,2024,01,08,10,2024-01,2496.120104
3,4,2024-04-13 23:50:35.462850,3514.40,944,purchase,Philadelphia,0,2024,04,13,23,2024-04,2480.872180
4,5,2024-07-12 18:51:35.462858,369.07,475,purchase,Phoenix,0,2024,07,12,18,2024-07,2508.055940


In [10]:
df['Mean_plus'] = df["Mean"]*bandwidth_plus
df['Mean_minus'] = df["Mean"]*bandwidth_minus

In [11]:
df['un_amount'] = np.where(((df['Amount'] < df['Mean_minus']) |(df['Amount'] > df['Mean_plus'])),1,0)
df['TransactionType'] = np.where(df['TransactionType']=='refund', 1, 0)

In [12]:
# Standardise the amounts
df['standardised_amount']= (df['Amount'] - df['Amount'].mean()) / df['Amount'].std()

In [13]:
df = pd.merge(df,fraud_rates_cities,on='Location')
df = pd.merge(df,fraud, on='MerchantID')
df = df.drop(columns=['TransactionID','TransactionDate','MerchantID','Location','Year-Month','Mean','Mean_plus','Mean_minus','Amount'])
df

,TransactionType,IsFraud,Year,Month,Day,hour,un_amount,standardised_amount,per_100k,FraudCount
0,1,0,2024,03,19,13,0,0.112740,8.0,1
1,0,0,2024,01,08,10,1,-1.187655,1.0,1
2,0,0,2024,07,12,18,1,-1.475318,6.0,1
3,1,0,2024,01,02,11,1,-0.714213,6.0,1
4,0,0,2024,05,12,12,1,-1.095303,1.0,2
...,...,...,...,...,...,...,...,...,...,...
63401,1,0,2024,01,12,17,0,0.268187,1.0,1
63402,0,0,2024,05,25,06,1,0.864416,3.0,2
63403,1,0,2024,04,20,15,1,-1.305499,6.0,1
63404,0,0,2024,10,18,09,1,0.869165,6.0,2


In [14]:
df.to_csv('fraud_data_cleaned.csv', index=False)